# 1 つの文書に複数のラベルを付ける (マルチラベル分類)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_classification/multilabel.ipynb)

NLP クックブック 09-11 ／ テキスト分類

1 つの記事が「経済」かつ「国際」のように、複数のカテゴリに同時に属する場合の分類をしたい。

In [ ]:
import nltk
nltk.download("reuters", quiet=True)
from nltk.corpus import reuters
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC

train_ids = [i for i in reuters.fileids() if i.startswith("training/")]
test_ids = [i for i in reuters.fileids() if i.startswith("test/")]
print(len(train_ids), len(test_ids), "カテゴリ数:", len(reuters.categories()))
print(reuters.categories(train_ids[1]), reuters.raw(train_ids[1])[:80].replace("\n", " "))

mlb = MultiLabelBinarizer()
Y_train = mlb.fit_transform([reuters.categories(i) for i in train_ids])   # 0/1 の行列 (文書数 × ラベル数)
Y_test = mlb.transform([reuters.categories(i) for i in test_ids])
vec = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=2)
X_train, X_test = vec.fit_transform(reuters.raw(i) for i in train_ids), vec.transform(reuters.raw(i) for i in test_ids)

In [ ]:
clf = OneVsRestClassifier(LinearSVC(C=1)).fit(X_train, Y_train)   # ラベルごとに 2 値分類器を学習
pred = clf.predict(X_test)
print("micro-F1:", f1_score(Y_test, pred, average="micro").round(3),
      " macro-F1:", f1_score(Y_test, pred, average="macro", zero_division=0).round(3),
      " 完全一致率:", (pred == Y_test).all(axis=1).mean().round(3))
for i in range(3):
    print("正解:", mlb.inverse_transform(Y_test[i:i + 1])[0], " 予測:", mlb.inverse_transform(pred[i:i + 1])[0])

- micro-F1 は頻出ラベルに、macro-F1 は稀なラベルに引っ張られる。両者の差が大きいのは、稀なラベルが当たっていない証拠。
- BERT でやる場合は `AutoModelForSequenceClassification(..., problem_type="multi_label_classification")` とし、ラベルを float の 0/1 ベクトルで与える (損失が BCE になる)。